In [0]:
sources = "licenses"
layer = "silver"

In [0]:
%run "../_micellanous/utils/init_environment_base"

In [0]:
%run "../_micellanous/utils/init_silver_environment"

In [0]:

display(source_to_load)

In [0]:
source_to_load.createOrReplaceTempView("source_to_load")

df_silver = spark.sql("""
    SELECT 
        socrata_id,
        application_id,
        license_number,

        json_struct.application_type AS application_type,
        json_struct.business_name AS business_name,
        json_struct.business_unique_id AS business_unique_id,
        json_struct.business_category AS business_category,
        json_struct.license_type AS license_type,
        json_struct.intake_channel AS intake_channel,
        json_struct.status AS status,

        CAST(json_struct.submission_date AS TIMESTAMP) AS submission_date,
        CAST(json_struct.date_closed AS TIMESTAMP) AS date_closed,

        json_struct.contact_phone AS contact_phone,
        json_struct.address_type AS address_type,
        json_struct.building_number AS building_number,
        json_struct.street AS street,
        json_struct.street_2 AS street_2,
        json_struct.unit_type AS unit_type,
        json_struct.unit AS unit,
        json_struct.city AS city,
        json_struct.state AS state,
        json_struct.zip AS zip,
        json_struct.borough AS borough,

        json_struct.community_board AS community_board,
        json_struct.council_district AS council_district,
        json_struct.bin AS bin,
        json_struct.bbl AS bbl,
        json_struct.nta AS nta,
        json_struct.census_block_2010_ AS census_block_2010,
        json_struct.census_tract_2010_ AS census_tract_2010,

        CAST(json_struct.latitude AS DOUBLE) AS latitude,
        CAST(json_struct.longitude AS DOUBLE) AS longitude,

        CAST(created_at AS TIMESTAMP) AS socrata_created_at,
        CAST(updated_at AS TIMESTAMP) AS socrata_updated_at,

        current_timestamp() AS silver_loaded_at

    FROM source_to_load
""")

df_silver = df_silver.na.replace(["NA", "null", ""], None)
df_silver.createOrReplaceTempView("vw_silver")

In [0]:
display(df_silver)

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {silver_full_tablename}
          USING DELTA
          CLUSTER BY AUTO
          TBLPROPERTIES (
              delta.enableChangeDataFeed = true,
              delta.enableDeletionVectors = true)
           AS
           SELECT * FROM  vw_silver WHERE 0 = 1  
          """)

In [0]:
merge_summary = spark.sql(f"""
          MERGE INTO {silver_full_tablename} AS target
          USING vw_silver AS source
          ON target.socrata_id = source.socrata_id
          WHEN MATCHED THEN UPDATE SET * 
          WHEN NOT MATCHED THEN INSERT *
          """)
display(merge_summary)          
